In [1]:
## Task 1: Dataset Understanding

import pandas as pd
from pathlib import Path

# Load the dataset
data_path = Path('..') / '..' / 'ai_project_synthetic_datasets' / 'part_3_nlp_sequence_modeling' / 'customer_support_text_classification.csv'
df = pd.read_csv(data_path)
# 

# Number of records
num_records = len(df)

# Target labels/classes
target_classes = df['sentiment_label'].unique().tolist()

# Sample text records
sample_records = df.head(3).to_dict('records')

# Average text length (assuming 'customer_message' is the text column, based on context)
if 'customer_message' in df.columns:
    df['text_length'] = df['customer_message'].apply(lambda x: len(str(x).split()))
    avg_text_length = df['text_length'].mean()
else:
    avg_text_length = "N/A"

# Class distribution
class_distribution = df['sentiment_label'].value_counts().to_dict()

print(f"Number of records: {num_records}")
print(f"Target classes: {target_classes}")
print(f"Sample records: {sample_records}")
print(f"Average text length: {avg_text_length}")
print(f"Class distribution: {class_distribution}")

Number of records: 1500
Target classes: ['neutral', 'positive', 'negative']
Sample records: [{'ticket_id': 'TKT00001', 'channel': 'chat', 'customer_message': 'I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.', 'sentiment_label': 'neutral', 'word_count': 18, 'urgent_flag': 1}, {'ticket_id': 'TKT00002', 'channel': 'phone', 'customer_message': 'I need information about the payment process.', 'sentiment_label': 'neutral', 'word_count': 7, 'urgent_flag': 0}, {'ticket_id': 'TKT00003', 'channel': 'email', 'customer_message': 'The refund process was fast and convenient. I appreciate the quick response.', 'sentiment_label': 'positive', 'word_count': 12, 'urgent_flag': 0}]
Average text length: 12.722666666666667
Class distribution: {'neutral': 524, 'negative': 497, 'positive': 479}


In [3]:
# Task 2 : Data Preprocessing

import re
import tensorflow as tf
#from tensorflow.keras.preprocessing.text import Tokenizer
#from tensorflow.keras.preprocessing.sequence import pad_sequences

# Dataset already loaded from previous cell, no need to reload
# (uses df defined in earlier cell)

# 1. Cleaning: Lowercasing and removing special characters
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Keep only letters and spaces
    return text

df['cleaned_text'] = df['customer_message'].apply(clean_text)

# 2. Tokenization using Keras (standard for sequence models)
max_vocab_size = 5000
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_text'])

# Convert to sequences of integers
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])

# 3. Padding/Truncating
max_length = 25  # Padding to 25 based on the average length we found (~12 words)
padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# Save intermediate dataset just in case
df.to_csv("customer_support_preprocessed.csv", index=False)

# Display a comparison for the first record
original_sample = df['customer_message'].iloc[0]
cleaned_sample = df['cleaned_text'].iloc[0]
sequence_sample = sequences[0]
padded_sample = padded_sequences[0].tolist()
word_index_len = len(tokenizer.word_index)

print(f"Original: {original_sample}")
print(f"Cleaned: {cleaned_sample}")
print(f"Tokenized Sequence: {sequence_sample}")
print(f"Padded Sequence: {padded_sample}")
print(f"Total Vocabulary Size: {word_index_len}")

Original: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Cleaned: i need information about the payment process my ticket number is  please respond as soon as possible
Tokenized Sequence: [5, 30, 137, 40, 2, 91, 34, 4, 7, 8, 3, 11, 13, 9, 14, 9, 15]
Padded Sequence: [5, 30, 137, 40, 2, 91, 34, 4, 7, 8, 3, 11, 13, 9, 14, 9, 15, 0, 0, 0, 0, 0, 0, 0, 0]
Total Vocabulary Size: 182
